# Model Latih 2: HOG dan SVM RBF

Notebook ini melatih model klasifikasi 24 huruf ASL statis menggunakan Histogram of Oriented Gradients (HOG) dan Support Vector Machine dengan kernel Radial Basis Function (SVM RBF). Metode ini berbeda dari model CNN utama dan Model Latih A.

Notebook dapat dijalankan dari atas ke bawah di Google Colab. Hasil evaluasi, grafik, confusion matrix, contoh prediksi, ukuran model, dan ringkasan untuk laporan akan disimpan ke folder output. Data uji resmi tidak digunakan untuk training atau pemilihan parameter.

In [ ]:
%pip -q install scikit-image scikit-learn opencv-python-headless joblib pandas matplotlib

from pathlib import Path
import hashlib
import json
import time
import urllib.request
import zipfile

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

SEED = 42
IMAGE_SIDE = 28
AUGMENTED_COPIES = 1
HOG_CONFIG = dict(orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2), block_norm='L2-Hys')
PCA_COMPONENTS = 128
SVM_C = 10.0
SVM_GAMMA = 'scale'
SVM_CACHE_MB = 4096

WORK = Path('/content/model_latih_2_hog_svm')
DATA = WORK / 'data'
ART = WORK / 'artifacts'
DATA.mkdir(parents=True, exist_ok=True)
ART.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = DATA / 'sign_mnist_train.csv'
TEST_CSV = DATA / 'sign_mnist_test.csv'
TRAIN_URL = 'https://raw.githubusercontent.com/gchilingaryan/Sign-Language/master/sign_mnist_train.csv'
TEST_URL = 'https://raw.githubusercontent.com/gchilingaryan/Sign-Language/master/sign_mnist_test.csv'
CLASS_NAMES = list('ABCDEFGHIKLMNOPQRSTUVWXY')
SOURCE_LABELS = [value for value in range(26) if value not in (9, 25)]
LABEL_TO_INDEX = {value: index for index, value in enumerate(SOURCE_LABELS)}

def download(url, path):
    if path.exists() and path.stat().st_size > 1000:
        print('Memakai cache', path.name)
        return
    request = urllib.request.Request(url, headers={'User-Agent': 'model-latih-2/1.0'})
    with urllib.request.urlopen(request, timeout=60) as response:
        path.write_bytes(response.read())
    print('Downloaded', path.name, path.stat().st_size, 'bytes')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

download(TRAIN_URL, TRAIN_CSV)
download(TEST_URL, TEST_CSV)

def load_dataset(path):
    frame = pd.read_csv(path)
    raw_labels = frame.iloc[:, 0].to_numpy(dtype=np.int64)
    if set(raw_labels.tolist()) - set(SOURCE_LABELS):
        raise ValueError('Dataset mengandung label J atau Z yang tidak boleh dipakai.')
    labels = np.asarray([LABEL_TO_INDEX[int(value)] for value in raw_labels], dtype=np.int64)
    images = frame.iloc[:, 1:].to_numpy(dtype=np.uint8).reshape(-1, IMAGE_SIDE, IMAGE_SIDE)
    if images.shape[1:] != (28, 28):
        raise ValueError(f'Ukuran gambar tidak sesuai: {images.shape}')
    return images, labels

images_all_train, labels_all_train = load_dataset(TRAIN_CSV)
images_test, labels_test = load_dataset(TEST_CSV)
assert images_all_train.shape == (27455, 28, 28)
assert images_test.shape == (7172, 28, 28)
assert len(np.unique(labels_all_train)) == 24 and len(np.unique(labels_test)) == 24

images_train, images_val, labels_train, labels_val = train_test_split(
    images_all_train, labels_all_train, test_size=0.20, random_state=SEED, stratify=labels_all_train
)

def augment_image(image, generator):
    angle = float(generator.uniform(-10, 10))
    zoom = float(generator.uniform(0.92, 1.08))
    shift_x = float(generator.uniform(-0.08, 0.08) * IMAGE_SIDE)
    shift_y = float(generator.uniform(-0.08, 0.08) * IMAGE_SIDE)
    shear = float(generator.uniform(-0.08, 0.08))
    matrix = cv2.getRotationMatrix2D((IMAGE_SIDE / 2, IMAGE_SIDE / 2), angle, zoom)
    matrix[0, 1] += shear
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y
    return cv2.warpAffine(image, matrix, (IMAGE_SIDE, IMAGE_SIDE), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

def make_augmented(images, labels, copies):
    generator = np.random.default_rng(SEED)
    if copies == 0:
        return images.copy(), labels.copy()
    copies_data = []
    for index in range(copies):
        copies_data.append(np.stack([augment_image(image, generator) for image in images]))
        print('Augmentation copy', index + 1, 'of', copies, 'done')
    output_images = np.concatenate([images] + copies_data)
    output_labels = np.concatenate([labels] * (copies + 1))
    order = generator.permutation(len(output_images))
    return output_images[order], output_labels[order]

images_train_aug, labels_train_aug = make_augmented(images_train, labels_train, AUGMENTED_COPIES)
split_summary = {
    'official_train': int(len(images_all_train)), 'train_before_augmentation': int(len(images_train)),
    'validation': int(len(images_val)), 'train_after_augmentation': int(len(images_train_aug)),
    'official_test': int(len(images_test)), 'augmentation_copies': AUGMENTED_COPIES,
    'seed': SEED, 'test_used_for_training': False,
}
config = {'method': 'HOG + SVM RBF', 'classes': CLASS_NAMES, 'split': split_summary,
          'hog': HOG_CONFIG, 'pca_components': PCA_COMPONENTS,
          'svm': {'kernel': 'rbf', 'C': SVM_C, 'gamma': SVM_GAMMA},
          'augmentation': {'rotation': [-10, 10], 'shift_fraction': [-0.08, 0.08],
                           'zoom': [0.92, 1.08], 'shear': [-0.08, 0.08], 'horizontal_flip': False},
          'dataset_sha256': {'train': sha256(TRAIN_CSV), 'test': sha256(TEST_CSV)}}
(ART / 'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(ART / 'split_summary.json').write_text(json.dumps(split_summary, indent=2), encoding='utf-8')
print(json.dumps(split_summary, indent=2))

In [ ]:
# Bukti visual dataset, distribusi kelas, dan augmentasi.
fig, axes = plt.subplots(2, 12, figsize=(15, 3))
for index, axis in enumerate(axes.flat):
    sample = np.flatnonzero(labels_test == index)[0]
    axis.imshow(images_test[sample], cmap='gray', vmin=0, vmax=255)
    axis.set_title(CLASS_NAMES[index], fontsize=9)
    axis.axis('off')
fig.suptitle('Contoh kelas pada Sign-MNIST test', y=1.03)
fig.tight_layout()
fig.savefig(ART / 'dataset_examples.png', dpi=220, bbox_inches='tight')
plt.show()

counts = np.bincount(labels_test, minlength=24)
fig, axis = plt.subplots(figsize=(11, 3.5))
axis.bar(CLASS_NAMES, counts, color='#2b6284')
axis.set(xlabel='Kelas huruf', ylabel='Jumlah gambar', title='Distribusi kelas Sign-MNIST test')
axis.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(ART / 'test_class_distribution.png', dpi=220)
plt.show()

generator = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
axes[0].imshow(images_train[0], cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Train asli')
for index in range(1, 5):
    axes[index].imshow(augment_image(images_train[0], generator), cmap='gray', vmin=0, vmax=255)
    axes[index].set_title(f'Augmentasi {index}')
for axis in axes: axis.axis('off')
fig.suptitle('Augmentasi train tanpa horizontal flip', y=1.03)
fig.tight_layout()
fig.savefig(ART / 'augmentation_examples.png', dpi=220, bbox_inches='tight')
plt.show()
print('Validation dan test tidak diaugmentasi:', images_val.shape, images_test.shape)

## 3. Ekstraksi HOG dan training SVM RBF

HOG mengambil pola arah tepi dan kontur tangan. PCA hanya di-fit pada train, lalu validation dan test hanya ditransformasi. SVM hanya dilatih pada train yang sudah diaugmentasi.

In [ ]:
def extract_hog(images):
    features = np.asarray([hog(image, visualize=False, **HOG_CONFIG) for image in images], dtype=np.float32)
    if not np.isfinite(features).all():
        raise ValueError('Fitur HOG berisi nilai tidak valid.')
    return features

started = time.perf_counter()
hog_train = extract_hog(images_train_aug)
hog_val = extract_hog(images_val)
hog_test = extract_hog(images_test)
print('Dimensi HOG:', hog_train.shape[1])
print('Waktu HOG:', round(time.perf_counter() - started, 2), 'detik')

components = min(PCA_COMPONENTS, hog_train.shape[0] - 1, hog_train.shape[1])
pca = PCA(n_components=components, whiten=True, svd_solver='randomized', random_state=SEED)
started = time.perf_counter()
pca_train = pca.fit_transform(hog_train).astype(np.float32)
pca_val = pca.transform(hog_val).astype(np.float32)
pca_test = pca.transform(hog_test).astype(np.float32)
print('PCA:', components, 'komponen, variansi kumulatif:', round(float(pca.explained_variance_ratio_.sum()), 5))
print('Waktu PCA:', round(time.perf_counter() - started, 2), 'detik')

svm_model = SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA, cache_size=SVM_CACHE_MB, probability=False, random_state=SEED)
started = time.perf_counter()
svm_model.fit(pca_train, labels_train_aug)
training_seconds = time.perf_counter() - started
print('Training SVM selesai:', round(training_seconds / 60, 2), 'menit')
print('Support vector:', int(len(svm_model.support_)))
assert len(svm_model.classes_) == 24

In [ ]:
def evaluate(name, features, labels):
    started = time.perf_counter()
    predictions = svm_model.predict(features)
    elapsed = time.perf_counter() - started
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {
        'split': name, 'samples': int(len(labels)),
        'correct': int((predictions == labels).sum()), 'incorrect': int((predictions != labels).sum()),
        'accuracy': float(accuracy_score(labels, predictions)), 'macro_precision': float(precision),
        'macro_recall': float(recall), 'macro_f1': float(f1), 'prediction_seconds': float(elapsed),
        'predictions': predictions, 'report': classification_report(labels, predictions, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
    }

validation_result = evaluate('validation', pca_val, labels_val)
test_result = evaluate('test_clean', pca_test, labels_test)
public_results = [{key: value for key, value in result.items() if key not in ('predictions', 'report')} for result in (validation_result, test_result)]
display(pd.DataFrame(public_results))
display(pd.DataFrame(test_result['report']).T)
(ART / 'metrics.json').write_text(json.dumps(public_results, indent=2), encoding='utf-8')
(ART / 'classification_report_test.json').write_text(json.dumps(test_result['report'], indent=2), encoding='utf-8')
pd.DataFrame(public_results).to_csv(ART / 'metrics.csv', index=False)

predictions_test = test_result['predictions']
prediction_table = pd.DataFrame({'row_test_zero_based': np.arange(len(labels_test)), 'row_test_one_based': np.arange(len(labels_test)) + 1,
                                 'actual': [CLASS_NAMES[int(value)] for value in labels_test],
                                 'predicted': [CLASS_NAMES[int(value)] for value in predictions_test]})
prediction_table['correct'] = prediction_table['actual'] == prediction_table['predicted']
prediction_table.to_csv(ART / 'test_predictions.csv', index=False)
prediction_table[~prediction_table['correct']].to_csv(ART / 'misclassified_test.csv', index=False)
print('Kesalahan test:', int((~prediction_table['correct']).sum()))

In [ ]:
# Confusion matrix dan metrik per kelas.
cm = confusion_matrix(labels_test, predictions_test, labels=np.arange(24))
fig, axis = plt.subplots(figsize=(8.5, 7.2))
image = axis.imshow(cm, cmap='Blues')
for row in range(24):
    for column in range(24):
        if cm[row, column] > 0:
            axis.text(column, row, str(cm[row, column]), ha='center', va='center', fontsize=6,
                      color='white' if cm[row, column] > cm.max() * 0.5 else 'black')
axis.set(xticks=np.arange(24), yticks=np.arange(24), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
         xlabel='Huruf prediksi', ylabel='Huruf sebenarnya', title=f'HOG + SVM RBF: {test_result["correct"]}/{test_result["samples"]} benar')
fig.colorbar(image, ax=axis, shrink=0.78, label='Jumlah gambar')
fig.tight_layout()
fig.savefig(ART / 'confusion_matrix_test.png', dpi=240)
plt.show()
np.save(ART / 'confusion_matrix_test.npy', cm)

per_class = pd.DataFrame(test_result['report']).T.loc[CLASS_NAMES, ['precision', 'recall', 'f1-score', 'support']]
per_class.to_csv(ART / 'per_class_metrics.csv')
fig, axis = plt.subplots(figsize=(12, 4))
x = np.arange(len(CLASS_NAMES))
width = 0.25
axis.bar(x - width, per_class['precision'] * 100, width, label='Precision')
axis.bar(x, per_class['recall'] * 100, width, label='Recall')
axis.bar(x + width, per_class['f1-score'] * 100, width, label='F1-score')
axis.set_xticks(x, CLASS_NAMES)
axis.set_ylim(0, 105)
axis.set(xlabel='Kelas huruf', ylabel='Nilai (%)', title='Metrik per kelas pada test clean')
axis.legend()
axis.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(ART / 'per_class_metrics.png', dpi=220)
plt.show()

In [ ]:
# Contoh prediksi benar dan salah.
correct_indices = np.flatnonzero(predictions_test == labels_test)
wrong_indices = np.flatnonzero(predictions_test != labels_test)
chosen = correct_indices[:6].tolist() + wrong_indices[:6].tolist()
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for axis in axes.flat: axis.axis('off')
for axis, index in zip(axes.flat, chosen):
    axis.imshow(images_test[index], cmap='gray', vmin=0, vmax=255)
    color = '#1b7f45' if predictions_test[index] == labels_test[index] else '#b33b2e'
    axis.set_title(f'Asli: {CLASS_NAMES[labels_test[index]]}\nPrediksi: {CLASS_NAMES[predictions_test[index]]}\nBaris: {index + 1}', fontsize=9, color=color)
fig.suptitle('Contoh prediksi pada data uji', y=1.02)
fig.tight_layout()
fig.savefig(ART / 'prediction_examples.png', dpi=220, bbox_inches='tight')
plt.show()
print('Contoh ditampilkan:', len(chosen))

## 6. Simpan model dan bukti eksperimen

Joblib menyimpan PCA dan SVM. Model ini bukan TFLite. Jika dipasang ke Flutter, ekstraksi HOG perlu dibuat di sisi aplikasi atau native code.

In [ ]:
bundle = {'method': 'HOG + SVM RBF', 'class_names': CLASS_NAMES, 'hog_config': HOG_CONFIG,
          'pca_components': int(components), 'pca': pca, 'svm': svm_model}
model_path = ART / 'model_latih_2_hog_svm_rbf.joblib'
joblib.dump(bundle, model_path, compress=3)

def predict_images(images):
    return svm_model.predict(pca.transform(extract_hog(images)).astype(np.float32))

timing_images = images_test[:min(1000, len(images_test))]
started = time.perf_counter()
timing_predictions = predict_images(timing_images)
inference_seconds = time.perf_counter() - started
assert len(timing_predictions) == len(timing_images)
model_info = {'model_size_bytes': int(model_path.stat().st_size), 'model_size_mib': float(model_path.stat().st_size / 1048576),
              'support_vectors': int(len(svm_model.support_)), 'timed_samples': int(len(timing_images)),
              'end_to_end_inference_seconds': float(inference_seconds), 'latency_ms_per_image': float(inference_seconds / len(timing_images) * 1000),
              'training_seconds': float(training_seconds)}
(ART / 'model_info.json').write_text(json.dumps(model_info, indent=2), encoding='utf-8')
print(json.dumps(model_info, indent=2))

def pct(value): return f'{value * 100:.4f}%'.replace('.', ',')
report = f"""# Model Latih 2: HOG dan SVM RBF

## Data dan metode

Kami menggunakan Sign-MNIST dengan 24 kelas gambar ASL statis. Dataset resmi memiliki {len(images_all_train):,} gambar train dan {len(images_test):,} gambar test. Data train dibagi menjadi {len(images_train):,} train dan {len(images_val):,} validation secara stratified dengan seed {SEED}. Setelah satu salinan augmentasi, train berisi {len(images_train_aug):,} sampel. Data test tidak digunakan saat training.

Augmentasi terdiri atas rotasi -10 sampai 10 derajat, pergeseran sampai 8 persen, zoom 0,92 sampai 1,08, dan shear -0,08 sampai 0,08. Horizontal flip tidak digunakan. HOG memakai {HOG_CONFIG}. PCA menghasilkan {components} komponen dengan variansi kumulatif {pca.explained_variance_ratio_.sum() * 100:.2f} persen. SVM memakai kernel RBF, C = {SVM_C}, dan gamma = {SVM_GAMMA}.

## Hasil run

| Split | Sampel | Accuracy | Precision macro | Recall macro | F1 macro |
| --- | ---: | ---: | ---: | ---: | ---: |
| Validation | {len(labels_val):,} | {pct(validation_result['accuracy'])} | {pct(validation_result['macro_precision'])} | {pct(validation_result['macro_recall'])} | {pct(validation_result['macro_f1'])} |
| Test clean | {len(labels_test):,} | {pct(test_result['accuracy'])} | {pct(test_result['macro_precision'])} | {pct(test_result['macro_recall'])} | {pct(test_result['macro_f1'])} |

Pada test clean, model benar pada {test_result['correct']:,} dari {test_result['samples']:,} gambar dan salah pada {test_result['incorrect']:,} gambar. Angka ini adalah hasil run notebook, bukan perkiraan.

## Artefak bukti

`metrics.csv`, `classification_report_test.json`, `confusion_matrix_test.png`, `per_class_metrics.png`, `prediction_examples.png`, `dataset_examples.png`, `augmentation_examples.png`, `test_predictions.csv`, `misclassified_test.csv`, `model_info.json`, dan `model_latih_2_hog_svm_rbf.joblib`.

## Batasan

Hasil ini mengukur Sign-MNIST clean, bukan kamera ponsel. Joblib bukan TFLite sehingga integrasi Flutter membutuhkan implementasi HOG dan SVM di aplikasi atau kode native. Perbandingan dengan model lain harus memakai split dan data uji yang sama.
"""
(ART / 'report_model_latih_2.md').write_text(report, encoding='utf-8')
print(report)

zip_path = WORK / 'model_latih_2_hog_svm_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ART.rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to(WORK))
print('ZIP:', zip_path)
print('Untuk download di Colab: from google.colab import files; files.download(str(zip_path))')